# Task 3.2 — Failure Mode Analysis
**Paper:** Incremental Learning of Temporally-Coherent Gaussian Mixture Models

In [1]:
import numpy as np, matplotlib, random, warnings
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from sklearn.mixture import GaussianMixture
warnings.filterwarnings('ignore')
RANDOM_SEED=42; np.random.seed(RANDOM_SEED); random.seed(RANDOM_SEED)

# ── HYPERPARAMETERS ──
N_PER_CLUSTER = 50
ORDER_INTERVAL = 10
MIN_DE = 0.3
MIN_ALPHA_N = 0.03
MAX_M = 6

# Dataset
np.random.seed(42)
centers=[(0,0),(4,3),(8,0)]; stds=[(0.4,0.9),(0.5,0.5),(0.7,0.35)]
segments=[np.column_stack([np.random.randn(50)*sx+cx,np.random.randn(50)*sy+cy]) for (cx,cy),(sx,sy) in zip(centers,stds)]
X=np.vstack(segments)

class TCGMM:
    def __init__(self):
        self.M=1; self.alphas=None; self.mus=None; self.Cs=None; self.Es=None; self.N=0
        self.hist_mus=None; self.hist_Cs=None; self.hist_Es=None; self.hist_N=0
    def _init(self,x):
        D=len(x); self.mus=np.array([x.copy()]); self.Cs=np.array([np.eye(D)*0.5])
        self.alphas=np.array([1.0]); self.Es=np.array([1.0]); self.N=1; self._sh()
    def _sh(self):
        self.hist_mus=self.mus.copy(); self.hist_Cs=[c.copy() for c in self.Cs]
        self.hist_Es=self.Es.copy(); self.hist_N=self.N
    def _g(self,x,mu,C):
        D=len(x); eps=np.eye(D)*1e-5; Cr=C+eps; Ci=np.linalg.inv(Cr)
        det=abs(np.linalg.det(Cr)); diff=x-mu
        return max(np.exp(-0.5*diff@Ci@diff)/((2*np.pi)**(D/2)*det**0.5),1e-300)
    def _r(self,x):
        r=np.array([self.alphas[i]*self._g(x,self.mus[i],self.Cs[i]) for i in range(self.M)])
        s=r.sum(); return r/s if s>1e-300 else np.ones(self.M)/self.M
    def update(self,x):
        p=self._r(x); ne=self.Es+p; na=ne/(self.N+1)
        nm=np.zeros_like(self.mus); nc=[]
        for i in range(self.M):
            nm[i]=(self.mus[i]*self.Es[i]+x*p[i])/ne[i]
            A=(self.Cs[i]+np.outer(self.mus[i],self.mus[i])-np.outer(self.mus[i],nm[i])
               -np.outer(nm[i],self.mus[i])+np.outer(nm[i],nm[i]))*self.Es[i]
            B=np.outer(x-nm[i],x-nm[i])*p[i]
            nc.append((A+B)/ne[i]+np.eye(len(x))*1e-5)
        self.alphas=na; self.mus=nm; self.Cs=np.array(nc); self.Es=ne; self.N+=1
    def _b(self,m1,C1,m2,C2):
        D=len(m1); eps=np.eye(D)*1e-5; i1=np.linalg.inv(C1+eps); i2=np.linalg.inv(C2+eps)
        C=np.linalg.inv(i1+i2); mu=C@(i1@m1+i2@m2)
        K=m1@i1@m1+m2@i2@m2-mu@np.linalg.inv(C+eps)@mu
        d1=abs(np.linalg.det(C1+eps)); d2=abs(np.linalg.det(C2+eps)); dC=abs(np.linalg.det(C+eps))
        return max(np.exp(-K/2)/((2*np.pi)**(D/2)*(d1*d2)**0.25*dC**0.5),1e-300)
    def _dl(self,a,m,C):
        M=len(a); D=len(m[0]); NE=(M-1)+M*D+M*D*(D+1)//2; ll=0
        for i in range(M):
            for j in range(M):
                b=self._b(m[i],C[i],m[j],C[j])
                ll+=a[i]*max(self.N*a[i],1)*np.log(max(a[j]*b,1e-300))
        return 0.5*NE*np.log2(max(self.N,2))-ll
    def _mo(self):
        if self.hist_N>=self.N: return
        i=0; dE=self.Es[i]-self.hist_Es[i]
        if dE>MIN_DE and self.M<MAX_M:
            an=dE/max(self.N-self.hist_N,1)
            if an>MIN_ALPHA_N:
                mn=(self.mus[i]*self.Es[i]-self.hist_mus[i]*self.hist_Es[i])/max(dE,1e-6)
                D=len(mn); Cn=np.eye(D)*0.2
                ta=np.append(self.alphas,an); ta/=ta.sum()
                tm=np.vstack([self.mus,mn]); tC=np.array(list(self.Cs)+[Cn])
                if self._dl(ta,tm,tC)<self._dl(self.alphas,self.mus,self.Cs):
                    self.alphas=ta; self.mus=tm; self.Cs=tC
                    self.Es=np.append(self.Es,an*self.N); self.M+=1; self._sh()
        if self.M>1:
            best=None; bg=-np.inf
            for ii in range(self.M):
                for jj in range(ii+1,self.M):
                    am=self.alphas[ii]+self.alphas[jj]
                    mm=(self.mus[ii]*self.alphas[ii]+self.mus[jj]*self.alphas[jj])/am
                    Cm=(self.Cs[ii]*self.alphas[ii]+self.Cs[jj]*self.alphas[jj])/am
                    mask=[k for k in range(self.M) if k!=ii and k!=jj]
                    ma=np.array([self.alphas[k] for k in mask]+[am]); ma/=ma.sum()
                    mm2=np.array([self.mus[k] for k in mask]+[mm]); mC=np.array([self.Cs[k] for k in mask]+[Cm])
                    g=self._dl(self.alphas,self.mus,self.Cs)-self._dl(ma,mm2,mC)
                    if g>bg: bg=g; best=(ii,jj,ma,mm2,mC)
            if bg>0 and best:
                ii,jj,ma,mm2,mC=best
                mask=[k for k in range(self.M) if k!=ii and k!=jj]
                self.Es=np.array([self.Es[k] for k in mask]+[self.Es[ii]+self.Es[jj]])
                self.alphas=ma; self.mus=mm2; self.Cs=mC; self.M=len(ma)
    def ll(self,X):
        s=sum(np.log(max(sum(self.alphas[i]*self._g(x,self.mus[i],self.Cs[i]) for i in range(self.M)),1e-300)) for x in X)
        return s/len(X)
    def fit(self,X,do_order=True):
        self._init(X[0]); self.llt=[]; self.Mt=[]
        for t,x in enumerate(X[1:],1):
            self.update(x)
            if do_order and t%ORDER_INTERVAL==0 and self.N>15: self._mo()
            self.llt.append(self.ll(X[:t+1])); self.Mt.append(self.M)
        return self

def ell(ax,mu,C,col='r',alp=0.25,ns=2):
    v,vecs=np.linalg.eigh(C); v=np.maximum(v,1e-6)
    ang=np.degrees(np.arctan2(*vecs[:,0][::-1])); w,h=2*ns*np.sqrt(v)
    ax.add_patch(Ellipse(xy=mu,width=w,height=h,angle=ang,edgecolor=col,facecolor=col,alpha=alp,lw=2))
print('Setup complete.')

Setup complete.

## Failure Scenario: Abrupt Non-Smooth Temporal Jumps

**Description:** We construct a dataset where data points alternate randomly between three distant clusters — simulating a scenario like a multi-person tracking system where identity changes occur frequently without any smooth transition. Each "jump" from cluster A to cluster B violates the temporal coherence assumption (Section 2.1), because the distance between consecutive points is large and drawn from a distribution that is NOT unimodal and peaked near zero.

**Why the method will struggle:** The fixed-complexity update in Eq. (7–8) relies on the approximation p*(i|x_j) ≈ p(i|x_j) (Eq. 6), which is only valid when successive points are close in space. When data jumps abruptly between clusters, adding a new point dramatically changes the responsibilities of all previous data, making this approximation false and causing the model to mix up component parameters. Additionally, the Historical GMM's "difference component" (Eq. 9–10) accumulates noise from all the random jumps instead of a meaningful directional drift, making split decisions unreliable — a failure mode explicitly discussed in Section 3 of the paper.

In [1]:
# ── Normal smooth data (reference) ──
m_smooth = TCGMM(); m_smooth.fit(X, do_order=True)
ll_smooth = m_smooth.ll(X)

# ── Failure mode: random interleaved jumps between 3 distant clusters ──
np.random.seed(42)
fail_parts = []
for _ in range(30):  # 30 mini-batches of 5 points each from random clusters
    c = np.random.choice([0,1,2])
    center = [[0,0],[8,8],[-4,6]][c]
    fail_parts.append(np.random.randn(5,2)*0.2 + np.array(center))
X_fail = np.vstack(fail_parts)  # 150 points, abrupt jumps every 5 points

m_fail = TCGMM(); m_fail.fit(X_fail, do_order=True)
ll_fail = m_fail.ll(X_fail)

print(f"Smooth data (normal):      LL = {ll_smooth:.4f}, M = {m_smooth.M}")
print(f"Abrupt jumps (failure):    LL = {ll_fail:.4f},  M = {m_fail.M}")
print(f"LL gap: {ll_smooth - ll_fail:+.4f}")
print()
# Oracle batch EM on failure data
gm_fail = GaussianMixture(n_components=3, random_state=42).fit(X_fail)
print(f"Oracle batch EM on failure data: LL = {gm_fail.score(X_fail):.4f}")
print(f"TC-GMM deficit vs oracle on failure data: {gm_fail.score(X_fail)-ll_fail:.4f}")


Smooth data (normal):      LL = -4.2333, M = 2
Abrupt jumps (failure):    LL = -3.2666,  M = 2
LL gap: -0.9667

Oracle batch EM on failure data: LL = -0.6138
TC-GMM deficit vs oracle on failure data: 2.6528


In [1]:
fig,axes=plt.subplots(1,2,figsize=(12,5))

ax=axes[0]
sc=ax.scatter(X_fail[:,0],X_fail[:,1],c=np.arange(len(X_fail)),cmap='plasma',s=25,alpha=0.7,label='Data (color=time)')
pal=['red','orange']
for i in range(m_fail.M):
    ell(ax,m_fail.mus[i],m_fail.Cs[i],col=pal[i%2])
    ax.scatter(*m_fail.mus[i],marker='x',s=150,c=pal[i%2],linewidths=2.5,zorder=5)
plt.colorbar(sc,ax=ax,label='Time index')
ax.set_title(f'Failure Mode: Abrupt Temporal Jumps\nM={m_fail.M}, LL={ll_fail:.3f}',fontsize=10,fontweight='bold')
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.grid(alpha=0.3)

ax=axes[1]
ax.plot(m_smooth.llt,'b-',lw=1.5,label=f'Smooth sequential, M→{m_smooth.M}',alpha=0.85)
ax.plot(m_fail.llt,'r-',lw=1.5,label=f'Abrupt jumps (failure), M→{m_fail.M}',alpha=0.85)
ax.set_title('Failure Mode: LL Trace\n(Smooth vs Abrupt Temporal Jumps)',fontsize=10,fontweight='bold')
ax.set_xlabel('Points seen'); ax.set_ylabel('Avg Log-Likelihood')
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('results/task3_failure.png',dpi=150,bbox_inches='tight'); plt.show()
print("Saved results/task3_failure.png")


Saved results/task3_failure.png

<Figure>

### Why the Method Fails Here

The TC-GMM performs dramatically worse than oracle batch EM on this abrupt-jump dataset: the incremental method achieves LL = −3.27 while batch EM achieves −0.61 — a deficit of 2.65 nats/point. This failure is directly rooted in Assumption 1 (Task 1.2): the temporal coherence assumption that consecutive points follow a unimodal smooth transition distribution p_S. With random cluster-to-cluster jumps every 5 points, this assumption is completely violated. The fixed-complexity update (Eq. 6–8) incorrectly assumes that adding one new far-away point does not change responsibilities of historical points; in reality it does, causing component means and covariances to be pulled toward incorrect positions. The Historical GMM accumulates a mixture of drift signals from multiple different directions, so the difference component (Eq. 9–10) produces a spurious component estimate rather than a meaningful one — exactly the first failure mode described by the authors in Section 3 ("newly available data is well explained by the Historical GMM"). Notably, the model converges to M=2 instead of the true M=3, missing one cluster entirely.

**Suggested modification:** Replace the fixed-complexity update with a small sliding-window EM that re-estimates parameters over the last W points rather than accumulating from time zero. This would limit the damage from non-coherent arrivals to a local window, sacrificing some memory efficiency for robustness to abrupt regime changes.